# Sửa ngữ pháp CoEdIT GEC — hardcore

Model scratch **chưa train**; dự đoán chỉ minh họa inference, không phải kết quả chất lượng. Bạn tự thiết kế tiền xử lý, thay model và viết training sau khi đọc luồng vào/ra này.

## 1. Load dữ liệu cơ bản

Tập inference: **nhóm câu GEC của official CoEdIT validation, cùng held-out hiện có; không phải official test**. Nguồn/revision trong SOURCE/DATASET_ID. Dữ liệu thô được giữ nguyên; thao tác nhóm/sắp ID chỉ để giữ giao thức mẫu của hai bản đầy đủ, không fit đặc trưng.

Chạy từ thư mục dự án, sửa DATA_DIR trước Run All. DATA_MODE="native" đọc nguồn đã khai báo; DATA_MODE="local_jsonl" đọc bản xuất **đúng nguồn**, schema: validation.jsonl: task, src, tgt, _id của CoEdIT gốc; chỉ task=gec, giữ instruction trong raw model input. Đường dẫn ảnh trong JSONL phải hợp lệ từ thư mục chạy. Nếu dùng nguồn HF cần cài datasets; COCO cần torchvision/pycocotools; TFDS cần kernel TensorFlow tương thích. Không tự thay dataset khi thiếu dữ liệu.

In [ ]:
from pathlib import Path
import hashlib
import io
import json
import random
import string
import unicodedata
import numpy as np
import torch
from torch import nn
from PIL import Image

SEED=42
BATCH_SIZE=16
MAX_INPUT_BYTES=4096  # Giới hạn đầu vào cố định của model; không đổi số mẫu inference.
MAX_OUTPUT_TOKENS=32
IMAGE_SIZE=64
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DATA_MODE="native"  # Chọn local_jsonl rõ ràng để dùng bản xuất đúng dataset/schema bên dưới.

DATA_DIR=Path('data/gec_coedit')
CACHE_DIR=Path("artifacts/gec_coedit/hardcore/data_cache")
INFERENCE_SPLIT='held_out'
SOURCE={'row': 4, 'task': 'Grammar Error Correction', 'dataset': 'grammarly/coedit', 'revision': 'e9a255c33ef910bc33a9d2b522653fa87521583e', 'url': 'https://huggingface.co/datasets/grammarly/coedit'}
DATASET_ID='grammarly/coedit'
DATASET_REVISION='e9a255c33ef910bc33a9d2b522653fa87521583e'
KNOWN_GEC_PREFIXES=['Fix all grammatical errors:', 'Fix disfluencies in the sentence:', 'Fix errors in this text:', 'Fix grammar errors in this sentence:', 'Fix grammar errors:', 'Fix grammar in the sentence:', 'Fix grammar in this sentence:', 'Fix grammar:', 'Fix grammatical errors in this sentence:', 'Fix grammatical errors:', 'Fix grammatical mistakes in this sentence:', 'Fix grammaticality in this sentence:', 'Fix grammaticality of the sentence:', 'Fix grammaticality:', 'Fix the grammar mistakes:', 'Fix the grammatical mistakes:', 'Grammar improvements:', 'Improve the grammar of this text:', 'Improve the grammaticality of this sentence:', 'Improve the grammaticality of this text:', 'Improve the grammaticality:', 'Make the sentence fluent:', 'Make the sentence grammatical:', 'Remove all grammatical errors from this text:', 'Remove grammar mistakes:', 'Remove grammatical mistakes:', 'Update to remove grammar errors:']


def read_jsonl(path):
    if not path.is_file():
        raise FileNotFoundError(f"Thiếu {path}. Cung cấp JSONL từ đúng dataset/revision/schema đã ghi; không dùng dữ liệu giả thay thế.")
    with path.open(encoding="utf-8") as handle:
        rows=[json.loads(line) for line in handle if line.strip()]
    if not rows:raise ValueError(f"Tập dữ liệu rỗng: {path}")
    return rows

In [ ]:
if DATA_MODE=="local_jsonl":rows=read_jsonl(DATA_DIR/"validation.jsonl")
elif DATA_MODE=="native":
    from datasets import load_dataset
    rows=load_dataset(SOURCE["dataset"],revision=SOURCE["revision"],split="validation",cache_dir=str(CACHE_DIR))
else:raise ValueError("DATA_MODE phải là native hoặc local_jsonl.")
# Chỉ canonical key dùng để nhóm/định danh đúng giao thức; src truyền model vẫn là chuỗi thô gồm instruction.
groups={}
for row in rows:
    if row["task"]!="gec":continue
    prefix=next((p for p in sorted(KNOWN_GEC_PREFIXES,key=len,reverse=True) if row["src"].startswith(p)),None)
    if prefix is None:raise ValueError("Instruction GEC chưa kiểm tra; cần xác minh nguồn trước khi nhóm.")
    key=" ".join(unicodedata.normalize("NFC",row["src"][len(prefix):]).split())
    group=groups.setdefault(key,{"src":row["src"],"canonical_source":key,"refs":[]})
    ref=" ".join(unicodedata.normalize("NFC",row["tgt"]).split())
    if ref not in group["refs"]:group["refs"].append(ref)
pool=list(groups.values())

# Canonical sort chỉ xác định ID cùng hai bản trước; raw src không bị sửa để tạo đặc trưng.
pool=sorted(pool,key=lambda row:(row.get("canonical_source"," ".join(unicodedata.normalize("NFC",row["src"]).split())),
    tuple(" ".join(unicodedata.normalize("NFC",ref).split()) for ref in row["refs"])))
records=[{**row,"id":f"test:{index}"} for index,row in enumerate(pool)]

assert records, "Không có mẫu test/validation để inference."
assert len({row["id"] for row in records})==len(records), "ID đầu vào bị trùng."
print("Tập inference:",INFERENCE_SPLIT,"; số mẫu:",len(records))

## 2. Định nghĩa một model và inference trực tiếp

Các lớp học đều khởi tạo ngẫu nhiên. Chỉ chuyển đầu vào tối thiểu bên trong model; không có tokenizer/vocabulary fit, hàm tiền xử lý riêng, loss, optimizer, training hay checkpoint. Giới hạn chiều đầu vào/đầu ra ở phần cấu hình là cố định và chỉ phục vụ model minh họa; toàn bộ mẫu của split vẫn được inference. Không truyền gold target vào forward.

In [ ]:
class RawTextModel(nn.Module):
    """Một model scratch nhận chuỗi thô; chưa có bất kỳ training nào."""
    def __init__(self):
        super().__init__()
        self.embedding=nn.Embedding(257,16)
        self.encoder=nn.RNN(16,32,batch_first=True)
        self.decoder=nn.RNNCell(16,32)
        self.head=nn.Linear(32,96)
    def forward(self,texts):
        results=[]
        for text in texts:
            # Chuyển byte cố định ngay trong model, không fit vocabulary/tokenizer.
            ids=list(str(text).encode("utf-8")[:MAX_INPUT_BYTES]) or [256]
            values=torch.tensor(ids,device=self.embedding.weight.device)
            _,hidden=self.encoder(self.embedding(values)[None])
            state=hidden[-1]
            token=torch.tensor([256],device=values.device)
            output=[]
            for _ in range(MAX_OUTPUT_TOKENS):
                state=self.decoder(self.embedding(token),state)
                index=int(self.head(state).argmax(-1).item())
                if index==95:break
                output.append(chr(32+index))
                token=torch.tensor([32+index],device=values.device)
            results.append("".join(output))
        return results

In [ ]:
model=RawTextModel().to(DEVICE)
model.eval()
predictions=[]
with torch.inference_mode():
    for start in range(0,len(records),BATCH_SIZE):
        batch=records[start:start+BATCH_SIZE]
        inputs=[row["src"] for row in batch]
        outputs=model(inputs)
        assert len(outputs)==len(batch), "Số dự đoán trong batch không khớp."
        predictions.extend([{'id':row["id"],'prediction':output} for row,output in zip(batch,outputs)])

assert len(predictions)==len(records)>0
assert [row['id'] for row in predictions]==[row["id"] for row in records]
assert len({row['id'] for row in predictions})==len(records)
assert all(isinstance(row['prediction'],str) for row in predictions)

print(f"Đã inference {len(predictions)} mẫu trên {INFERENCE_SPLIT} bằng model scratch CHƯA TRAIN.")
print("Đầu ra minh họa thực thi, chưa đo chất lượng; cần tự xây dựng tiền xử lý, model và training.")
print(predictions[:2])
# File dự đoán chỉ được tạo khi người dùng chạy notebook; không cần checkpoint/helper.
PREDICTION_PATH=Path("artifacts/gec_coedit/hardcore/predictions.json")
PREDICTION_PATH.parent.mkdir(parents=True,exist_ok=True)
PREDICTION_PATH.write_text(json.dumps(predictions,ensure_ascii=False,indent=2),encoding="utf-8")